# Reduce and run cells from AllenDB

In [1]:
# !pip install --upgrade numpy==1.24.4 pandas==2.2.2 scipy==1.11.3> /dev/null 2>&1

# import os
# os.kill(os.getpid(), 9)#restart so the above packages can be used

In [2]:
# If running in Colab
!pip install --upgrade pip2 > /dev/null 2>&1
!git clone -b release_candidate https://github.com/V-Marco/ACT.git > /dev/null 2>&1

#choose specimen id and model type

In [3]:
# Mouse L2/3 PV cell
# https://celltypes.brain-map.org/experiment/electrophysiology/484635029
specimen_id = 484635029

model_type = 'perisomatic'#or 'all active'
work_dir = 'OriginalFromAllenDB'

In [4]:
from allensdk.api.queries.biophysical_api import BiophysicalApi
from allensdk.model.biophys_sim.config import Config
from allensdk.model.biophysical.utils import Utils
from neuron import h
import os
import json
import matplotlib.pyplot as plt
import math
import numpy as np
import sys
sys.path.append("ACT")
from act.passive import ACTPassiveModule

# If error, restart env

--No graphics will be displayed.
/home/hrbncv/miniconda3/envs/BMTK/lib/python3.9/site-packages/torch/__init__.py:1240: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


#Also if using hoc, double check parameters every time build_cell() is used

In [5]:
#get the model id

bp = BiophysicalApi()#using AllenDB's API
models = bp.get_neuronal_models(specimen_id)

for model in models:
  if model_type in model['name'].lower():
    print(f"model id = {model['id']}\n")
    model_id = model['id']
models

model id = 485602029



[{'id': 485602029,
  'name': 'Biophysical - perisomatic_Pvalb-IRES-Cre;Ai14-201791.05.01.01',
  'neuron_reconstruction_id': 496079599,
  'neuronal_model_template_id': 329230710,
  'specimen_id': 484635029},
 {'id': 496538965,
  'name': 'Biophysical - all active_Pvalb-IRES-Cre;Ai14-201791.05.01.01',
  'neuron_reconstruction_id': 496079599,
  'neuronal_model_template_id': 491455321,
  'specimen_id': 484635029}]

## Build the cell

add vecstim, cpampain.mod,NMDAIN.mod from https://github.com/yzerlaut/pv-sst-dendrites/tree/main/detailed_model and compile

if you need additonal modfiles from a public repo, provide a list of the names of the modfiles and a link to the folder

In [6]:
# Compile the modfiles
os.chdir(work_dir)
!nrnivmodl modfiles > /dev/null 2>&1

In [7]:
import sys, os
from functools import wraps

def suppress_output(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        devnull = open(os.devnull, 'w')
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout = sys.stderr = devnull
        try:
            return fn(*args, **kwargs)
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr
            devnull.close()
    return wrapper

**NOTE:** Control the soma area in the `build_cell` function below.

In [8]:
@suppress_output
def build_cell(soma_diam_multiplier = 1,using_hoc=False,hoc_filename=None,cell_name=None):

    if using_hoc:
      from neuron import h
      h.load_file(hoc_filename)
      return getattr(h, cell_name)()
    # Create the h object
    description = Config().load('manifest.json')
    utils = Utils(description)
    h = utils.h
    # Convert all "value" attributes to floats
    for dict in utils.description.data['genome']:
        for key, value in dict.items():
            if key == 'value': dict[key] = float(value)
    # Configure morphology
    morphology_path = description.manifest.get_path('MORPHOLOGY')
    utils.generate_morphology(morphology_path.encode('ascii', 'ignore'))
    utils.load_cell_parameters()
    # To match PP
    h.soma[0].diam = h.soma[0].diam * soma_diam_multiplier

    return h

In [9]:
soma_diam_multiplier = 6
hobject = build_cell(soma_diam_multiplier)
h.define_shape()

1.0

In [10]:
print(f"Soma area: {hobject.soma[0](0.5).area()}")
print(f"Soma diam: {hobject.soma[0].diam}")
print(f"Soma L: {hobject.soma[0].L}")

# print(f"Axon area: {hobject.axon[0](0.5).area()}")
# print(f"Axon diam: {hobject.axon[0].diam}")
# print(f"Axon L: {hobject.axon[0].L}")

Soma area: 4017.876017303496
Soma diam: 87.59880065917969
Soma L: 14.599853515625


Cell Morphology

In [11]:
soma_roots = hobject.soma[0].children()
print(f'Root dendrites: {soma_roots}')

# h.topology()

Root dendrites: [axon[0], dend[36], dend[31], dend[24], dend[23], dend[22], dend[0]]


Dendrite Selection Automation

In [ ]:
proximal_region = [20, 60]#um
distal_region = [160,200]

In [ ]:
# find the first segment in a branch that is in the prox range, then do the same for distal
#dict key is the sec name, the fields are the segment location, the distance from the soma to that seg, and the length of the section
h.distance(0, 0.5, sec=hobject.soma[0])

proximal_dict = {}
distal_dict = {}

for child_sec in hobject.soma[0].children():
    key = 0
    for sec in child_sec.subtree():
        for seg in sec:
          #because h.distance() returns distance to the section not the segment, and we
          #assume identical lengths for the segments, to location of the segment's
          #proximal end is h.distance + numseg * seglength

            dist = h.distance(sec(0)) + seg.x * sec.L

            data = {
                "root" :child_sec.name(),
                "sec": sec.name(),
                "seg": round(seg.x,2),
                "dist": round(dist,2),
                "L": round(sec.L,2)
                }

            key += 1

            if proximal_region[0] <= dist <= proximal_region[1]:

               proximal_dict[key]  = data


            if distal_region[0] <= dist <= distal_region[1]:

                distal_dict[key] = data



In [ ]:
for key in proximal_dict:
  print(proximal_dict[key])

for key in distal_dict:
  print(distal_dict[key])

common = []

for key in proximal_dict:
  root = proximal_dict[key]['root']
  for item in distal_dict:
    if distal_dict[item]['root'] == root:
      common.append(root)

common = list(set(common))

print(f"roots of the dendrites that cover both the proximal and distal regions:\n\t {common}")

In [ ]:
key = 0
print("proximal")

commonproxdict = {}
for key in proximal_dict:
  if proximal_dict[key]['root'] == common[0]:
    print(proximal_dict[key])
    commonproxdict.update({key:proximal_dict[key]})

    # print(f"{proximal_dict[key]['sec']}({proximal_dict[key]['seg']})")

print("distal:")
commondistaldict = {}
for key in distal_dict:
  if distal_dict[key]['root'] == common[0]:
    print(distal_dict[key])
    commondistaldict.update({key:distal_dict[key]})


Section Details

In [12]:
from neuron import h

def segment_properties(sec):
    """
    Return a list of dicts—one per segment in `sec`—with these fields:
      • index         : integer (0‐based)
      • x_center      : normalized center (0–1)
      • x_start       : normalized left edge (0–1)
      • x_end         : normalized right edge (0–1)
      • phys_start    : physical start distance from section origin (µm)
      • phys_center   : physical center distance from section origin (µm)
      • phys_end      : physical end distance from section origin (µm)
      • seg_length    : physical segment length = sec.L / sec.nseg (µm)
      • diam_center   : diameter at segment center (µm)
      • diam_start    : diameter at x_start
      • diam_end      : diameter at x_end
    """
    nseg = int(sec.nseg)
    L    = sec.L   # total physical length of this section (µm)
    seg_len = L / nseg

    props = []
    for i, seg in enumerate(sec):
        # normalized coordinates
        x_c      = seg.x              # (i + 0.5) / nseg
        x_start  = i      / nseg
        x_end    = (i + 1)/ nseg

        # physical distances along this section
        phys_start  = x_start * L
        phys_center = x_c     * L
        phys_end    = x_end   * L

        # diameters
        d_center = seg.diam
        d_start  = sec(x_start).diam
        d_end    = sec(x_end).diam

        props.append({
            'index':       i,
            'x_center':    x_c,
            'x_start':     x_start,
            'x_end':       x_end,
            'phys_start':  phys_start,
            'phys_center': phys_center,
            'phys_end':    phys_end,
            'seg_length':  seg_len,
            'diam_center': d_center,
            'diam_start':  d_start,
            'diam_end':    d_end
        })

    for p in props:
        print(
            f"seg#{p['index']:2d}  "
            f"x∈[{p['x_start']:.3f}–{p['x_end']:.3f}]  "
            f"phys∈[{p['phys_start']:.1f}–{p['phys_end']:.1f}]µm  "
            f"len={p['seg_length']:.2f}µm  "
            f"diam_center={p['diam_center']:.4f}µm  "
            f"diam_edges=[{p['diam_start']:.4f}, {p['diam_end']:.4f}]µm"
            )
            
    return props


In [13]:
from neuron import h

def section_avg_diameter(sec) -> float:
    """
    Compute the length-weighted average diameter of a single Section.

    That is, 
        ⎛  ∑ (diameter at each segment center × segment_length)  ⎞
    D =  ⎜ ----------------------------------------------------  ⎟,
        ⎝   total_length_of_section                          ⎠

    where segment_length = sec.L / sec.nseg.

    Parameters
    ----------
    sec : h.Section
        Any section whose diameter may vary (e.g. by PT3D points).

    Returns
    -------
    float
        The average diameter (µm) of that section.
    """
    # Ensure NEURON has built the piecewise diameter from PT3D data:
    h.define_shape()

    L    = sec.L              # total physical length of this section (µm)
    nseg = int(sec.nseg)      # number of isopotential compartments
    seg_len = L / nseg        # each segment’s length (µm)

    # Sum(diam_center * seg_len) over all segments:
    running = 0.0
    for seg in sec:
        running += seg.diam * seg_len

    # Divide by total length → average diameter
    return running / L


def branch_avg_diameter(root_sec) -> float:
    """
    Compute the length-weighted average diameter over an entire subtree.

    This walks every Section in root_sec.subtree(), then over each segment
    in those sections, sums (diam_center × seg_length), and divides by 
    the total cable length of the branch.

    Parameters
    ----------
    root_sec : h.Section
        The “root” of the branch (e.g. cell.dend[0]).

    Returns
    -------
    float
        The average diameter (µm) across all segments in the subtree.
    """
    # Ensure NEURON has built the piecewise diameter from PT3D data:
    h.define_shape()

    # First compute total length of the branch (sum of all Section lengths)
    total_length = 0.0
    for sec in root_sec.subtree():
        total_length += sec.L

    if total_length == 0:
        return 0.0  # avoid division by zero if a degenerate section

    # Now sum (diam_center × seg_length) over every segment in every section
    weighted_sum = 0.0
    for sec in root_sec.subtree():
        L    = sec.L
        nseg = int(sec.nseg)
        seg_len = L / nseg

        for seg in sec:
            weighted_sum += seg.diam * seg_len

    return weighted_sum / total_length


In [14]:
def subtree_surface_area(root_sec) -> float:
    """
    Total membrane surface area (µm²) of `root_sec` *and every section
    that branches from it*.

    Parameters
    ----------
    root_sec : h.Section
        The section that docks to the soma, e.g. cell.dend[0]

    Returns
    -------
    float
        Sum of areas of all segments in the subtree.
    """
    # Make sure NEURON has turned any pt3d data into segment geometry
    h.define_shape()

    area_total = 0.0
    for sec in root_sec.subtree():        # includes root_sec itself
        for seg in sec:                   # every segment in that section
            # seg.x is this segment's centre (0‒1); area() at that x
            # returns *this segment’s* surface area
            area_total += h.area(seg.x, sec=sec)

    return area_total


In [15]:
from neuron import h

def electrotonic_distance_sectionwise(target_sec,
                                      target_x: float,
                                      soma_ref,
                                      freq: float = 100.0) -> float:
    """
    Approximate the electrotonic distance X from soma_ref(0.5) to (target_sec, target_x)
    by summing (sec.L / lambda_sec) for each entire Section on the path,
    then subtracting the tail of the final section.

    Parameters
    ----------
    target_sec : h.Section
        Section containing the endpoint.
    target_x   : float
        Normalized location (0–1) in target_sec where distance ends.
    soma_ref   : h.Section
        Section to use as the origin (e.g. hobject.soma[0]).
    freq       : float
        Frequency (Hz) at which lambda_f is evaluated (default = 100).

    Returns
    -------
    X_total : float
        Approximate electrotonic distance (unitless, in λ units at `freq` Hz).
    """
    # 1) Anchor the distance origin at soma_ref(0.5)
    h.distance(0, 0.5, sec=soma_ref)

    # 2) Build the list of sections from the soma up to target_sec
    path_secs = []
    sec = target_sec
    while True:
        path_secs.insert(0, sec)
        parent_seg = sec.parentseg()
        if parent_seg is None:
            break
        sec = parent_seg.sec

    # 3) Sum sec.L / λ_section for each section in path_secs
    X_total = 0.0
    for sec in path_secs:
        sec.push()                   # make `lambda_f` refer to this section
        lam_sec = h.lambda_f(freq)   # λ at the center of sec (locations default to 0.5)
        h.pop_section()

        if lam_sec <= 0:
            raise RuntimeError(f"Non-positive lambda ({lam_sec}) in section {sec.name()}")

        X_total += sec.L / lam_sec

    # 4) Subtract the “unused” tail of the final section (since we only go to target_x)
    #    Compute λ_target exactly at (target_x) in target_sec:
    target_sec.push()
    lam_target = h.lambda_f(freq)  # λ at the middle of the section by default
    h.pop_section()

    # But we really want λ at `target_x`. For a quick fix we’ll reuse lam_target,
    # accepting a small error. If you want λ exactly at target_x, you’d need a
    # more advanced approach (e.g. manually compute λ = sqrt(rm/ra)*sqrt(diam/4) at target_x).

    # Physical “unused” length of target_sec from target_x to x=1.0
    unused_length = (1.0 - target_x) * target_sec.L
    X_total -= unused_length / lam_target

    return X_total





def electrotonic_distance_segmentwise(target_sec,
                                      target_x: float,
                                      soma_ref,
                                      freq: float = 100.0) -> float:
    """
    Compute X = ∫ ds/λ(s) from soma_ref(0.5) to (target_sec, target_x)
    by summing across individual segments, using each section’s λ at its midpoint.

    Parameters
    ----------
    target_sec : h.Section
        The section containing the endpoint.
    target_x   : float
        Normalized location (0–1) in target_sec where distance ends.
    soma_ref   : h.Section
        Section to use as the origin for distance() (e.g. cell.soma[0]).
    freq       : float
        Frequency (Hz) at which lambda_f is evaluated.

    Returns
    -------
    X_total : float
        Electrotonic distance (dimensionless, in λ units at `freq` Hz).
    """
    # 1) Ensure geometry is up-to-date
    h.define_shape()

    # 2) Anchor distance origin at soma_ref(0.5)
    h.distance(0, 0.5, sec=soma_ref)

    # 3) Build the list of sections from soma_ref → target_sec
    sec_path = []
    sec = target_sec
    while True:
        sec_path.insert(0, sec)
        parent_seg = sec.parentseg()
        if parent_seg is None:
            break
        sec = parent_seg.sec

    X_total = 0.0

    # 4) Walk each section in path_secs
    for sec in sec_path:
        nseg    = int(sec.nseg)
        seg_len = sec.L / nseg  # physical length of each segment in this section (µm)

        # Get λ for this section at its midpoint (x=0.5)
        sec.push()
        lam_sec = h.lambda_f(freq)   # λ at section midpoint (µm)
        h.pop_section()
        if lam_sec <= 0:
            raise RuntimeError(f"Nonpositive λ={lam_sec} in section {sec.name()}")

        # If this is not the final section, add all segments in full
        if sec is not target_sec:
            X_total += (sec.L / lam_sec)
        else:
            # In the final section, only add segments up to target_x
            seg_index_target = int(target_x * nseg)
            if seg_index_target >= nseg:
                seg_index_target = nseg - 1

            # 4a) Sum all *full* segments before the target segment
            X_total += (seg_index_target * seg_len) / lam_sec

            # 4b) Partial segment (the one containing target_x):
            #     length from segment start to target_x
            phys_target    = target_x * sec.L
            phys_seg_start = (seg_index_target / nseg) * sec.L
            partial_len    = phys_target - phys_seg_start

            X_total += partial_len / lam_sec

    return X_total



In [16]:
secn = 37
sec = hobject.dend[secn]

h.distance(0, 0.5, sec=hobject.soma[0])
obj = sec
dist = h.distance(obj(1))

print(f'Center distance from soma: {dist:.2f} um')
print(f"{sec} length: {sec.L} um")
print(f'Segments: {int(sec.nseg)}')

avg_sec = section_avg_diameter(sec)
print(f"Average diameter of {sec.name()}: {avg_sec:.4f} µm")

props = segment_properties(sec)

Center distance from soma: 209.94 um
dend[37] length: 184.91410311556683 um
Segments: 9
Average diameter of dend[37]: 0.2903 µm
seg# 0  x∈[0.000–0.111]  phys∈[0.0–20.5]µm  len=20.55µm  diam_center=0.3512µm  diam_edges=[0.3512, 0.3150]µm
seg# 1  x∈[0.111–0.222]  phys∈[20.5–41.1]µm  len=20.55µm  diam_center=0.3150µm  diam_edges=[0.3150, 0.2746]µm
seg# 2  x∈[0.222–0.333]  phys∈[41.1–61.6]µm  len=20.55µm  diam_center=0.2746µm  diam_edges=[0.2746, 0.2746]µm
seg# 3  x∈[0.333–0.444]  phys∈[61.6–82.2]µm  len=20.55µm  diam_center=0.2746µm  diam_edges=[0.2746, 0.2848]µm
seg# 4  x∈[0.444–0.556]  phys∈[82.2–102.7]µm  len=20.55µm  diam_center=0.2848µm  diam_edges=[0.2848, 0.2853]µm
seg# 5  x∈[0.556–0.667]  phys∈[102.7–123.3]µm  len=20.55µm  diam_center=0.2853µm  diam_edges=[0.2853, 0.2841]µm
seg# 6  x∈[0.667–0.778]  phys∈[123.3–143.8]µm  len=20.55µm  diam_center=0.2841µm  diam_edges=[0.2841, 0.2746]µm
seg# 7  x∈[0.778–0.889]  phys∈[143.8–164.4]µm  len=20.55µm  diam_center=0.2746µm  diam_edges=[0.27

Section Morphology

In [17]:
# sec = 37

print(f'Path to soma: NOT FINISHED')
print(f'Children branches: {sec.children()}')
print(f'Full subtree: {sec.subtree()}')
print(f"Subsections: {len(sec.subtree())}")

# 2) Average diameter of the entire branch (sec plus all its children):
avg_branch = branch_avg_diameter(sec)
print(f"Average diameter of branch rooted at {sec.name()}: {avg_branch:.4f} µm")

# 3) Total Surface Area of the entire branch (sec plus all its children):
branch_area = subtree_surface_area(sec)
print(f"Surface area of branch rooted at {sec.name()}: "
      f"{branch_area:.1f} µm²")


freq_test = 20
# Suppose you want electrotonic distance to 30% along that section:
# X_elec = electrotonic_distance_sectionwise(sec, 0.3, soma_ref=hobject.soma[0], freq=freq_test)
# print(f"Electrotonic distance ≈ {X_elec:.5f} (λ units at {freq_test} Hz)")

# Suppose you want e-distance to 100% along that section:
X_elec_segwise = electrotonic_distance_segmentwise(
    sec, 1.0, soma_ref=hobject.soma[0], freq=freq_test)
print(f"Electrotonic distance (segmentwise) ≈ {X_elec_segwise:.5f} (λ units at {freq_test} Hz)")


Path to soma: NOT FINISHED
Children branches: []
Full subtree: [dend[37]]
Subsections: 1
Average diameter of branch rooted at dend[37]: 0.2903 µm
Surface area of branch rooted at dend[37]: 168.6 µm²
Electrotonic distance (segmentwise) ≈ 0.33683 (λ units at 20 Hz)


AUTOMATION

In [18]:
primary_roots = []
stump_props = {}

for sec in soma_roots:
    # print(sec)

    branch_area = subtree_surface_area(sec)
    avg_branch = branch_avg_diameter(sec)
    X_elec_segwise = electrotonic_distance_segmentwise(sec, 0.43, soma_ref=hobject.soma[0], freq=freq_test)


    
    stump_props[sec.name()] = {
                            'subsecs': len(sec.subtree()), 
                            't_SA': branch_area, 
                            'Davg': avg_branch, 
                            'ED': X_elec_segwise, 
                            'red': True,
                            }
    
    if sec.name() in primary_roots:
        primary_branch_root = sec
        stump_props[sec.name()]['red'] = False

    print(f'{sec.name()}: {stump_props[sec.name()]}')
# print(stump_props)

axon[0]: {'subsecs': 2, 't_SA': 188.49555921538757, 'Davg': 1.0, 'ED': 0.012927395608806087, 'red': True}
dend[36]: {'subsecs': 5, 't_SA': 427.47406054615607, 'Davg': 0.32196122749605355, 'ED': 0.013076666323668785, 'red': True}
dend[31]: {'subsecs': 5, 't_SA': 141.68768313088813, 'Davg': 0.3451235686648551, 'ED': 0.005187909646130761, 'red': True}
dend[24]: {'subsecs': 7, 't_SA': 244.21663853080358, 'Davg': 0.30860850722005806, 'ED': 0.012231741098317353, 'red': True}
dend[23]: {'subsecs': 1, 't_SA': 80.80986341162082, 'Davg': 0.34466203858745337, 'ED': 0.051027863951114076, 'red': True}
dend[22]: {'subsecs': 1, 't_SA': 68.00742400072872, 'Davg': 0.2784035666867971, 'ED': 0.05812692021700003, 'red': True}
dend[0]: {'subsecs': 22, 't_SA': 1014.1261411691552, 'Davg': 0.32280255486172127, 'ED': 0.0065231643066797785, 'red': True}
